# SHAP Explainability Report for Diabetes Prediction

This notebook explains and demonstrates the SHAP feature added to this project. The goal is to make the diabetes prediction model more interpretable, so we can understand which input features influenced the model's predictions.

**Project context:** binary classification for diabetes prediction.

- `Outcome = 0`: non-diabetic / negative class
- `Outcome = 1`: diabetic / positive class

The SHAP feature is implemented mainly through two files:

- `shap_demo.py`: runnable demonstration script
- `shap_explainability.py`: reusable SHAP helper function

## 1. What Is SHAP?

SHAP stands for **SHapley Additive exPlanations**. It is an explainable AI method based on Shapley values from game theory.

The main idea is simple: a model prediction can be treated like a team result, where each feature contributes something to the final decision.

For a diabetes prediction, the model prediction can be explained like this:

```text
base prediction
+ effect of Glucose
+ effect of BMI
+ effect of Age
+ effect of Pregnancies
+ effect of other features
= final prediction
```

A positive SHAP value pushes the prediction toward the positive class, which is diabetes in this project. A negative SHAP value pushes the prediction toward the negative class.

## 2. Why SHAP Is Useful In This Healthcare Project

Accuracy alone is not enough in healthcare-related machine learning. If a model predicts that a patient has a higher diabetes risk, we should also understand which clinical features influenced that prediction.

SHAP helps this project by answering questions such as:

- Which features are most important overall?
- Does `Glucose` strongly affect diabetes predictions?
- Are the model explanations medically reasonable?
- Can we explain the model's behavior in a report instead of only showing accuracy?

Important limitation: SHAP explains the model's behavior. It does **not** prove medical causality.

## 3. Import Libraries

This cell imports the required libraries and the reusable SHAP helper function from `shap_explainability.py`.

In [ ]:
import matplotlib
matplotlib.use("Agg")

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from shap_explainability import apply_shap

## 4. Load The Diabetes Dataset

The dataset is loaded from `dataset/diabetes.csv`. The target column is `Outcome`, and all other columns are used as input features.

In [ ]:
data = pd.read_csv("dataset/diabetes.csv")

target = "Outcome"
x = data.drop(target, axis=1)
y = data[target]

print("Dataset shape:", data.shape)
print("Feature columns:", list(x.columns))
data.head()

## 5. Split And Scale The Data

The dataset is split into training and testing sets. The features are then standardized using `StandardScaler`.

Scaling is important for Logistic Regression because features have different ranges. For example, `Insulin` can be much larger numerically than `DiabetesPedigreeFunction`.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

print("Training samples:", x_train.shape[0])
print("Testing samples:", x_test.shape[0])

## 6. Train The Logistic Regression Model

This demo uses Logistic Regression because it is simple, fast, and interpretable. The model learns relationships between the input features and the diabetes outcome.

In [ ]:
model = LogisticRegression(random_state=100, max_iter=1000)
model.fit(x_train_scaled, y_train)

y_predict = model.predict(x_test_scaled)

print(f"Demo accuracy: {accuracy_score(y_test, y_predict):.4f}")
print(classification_report(y_test, y_predict))

## 7. Apply SHAP

This is the main SHAP call. It sends the trained model, training data, testing data, and feature names into `apply_shap(...)`.

The two most important export parameters are:

- `output_dir="shap_outputs/logistic_regression_demo"`: where files are saved
- `plot_name="logistic_regression_demo"`: filename prefix for the exported files

In [ ]:
shap_result = apply_shap(
    model,
    x_train_scaled,
    x_test_scaled,
    feature_names=x.columns,
    output_dir="shap_outputs/logistic_regression_demo",
    plot_name="logistic_regression_demo",
    max_explain_samples=100,
    show_plots=False,
)

shap_result["feature_importance"]

## 8. Exported SHAP Outputs

The `apply_shap(...)` function saves three report artifacts:

```text
shap_outputs/logistic_regression_demo/logistic_regression_demo_shap_summary.png
shap_outputs/logistic_regression_demo/logistic_regression_demo_shap_bar.png
shap_outputs/logistic_regression_demo/logistic_regression_demo_shap_importance.csv
```

The saving happens inside `shap_explainability.py`:

- summary plot path is created with `summary_path = output_path / f"{plot_name}_shap_summary.png"`
- bar plot path is created with `bar_path = output_path / f"{plot_name}_shap_bar.png"`
- CSV path is created with `importance_path = output_path / f"{plot_name}_shap_importance.csv"`

## 9. SHAP Summary Plot

The summary plot shows feature importance and feature effect direction. Each point represents one sample. Features higher on the plot are more important.

![SHAP summary plot](shap_outputs/logistic_regression_demo/logistic_regression_demo_shap_summary.png)

## 10. SHAP Bar Plot

The bar plot gives a simpler global feature-importance ranking using mean absolute SHAP value.

![SHAP bar plot](shap_outputs/logistic_regression_demo/logistic_regression_demo_shap_bar.png)

## 11. Current Demo Result

When the demo was run, the model produced this result:

```text
Demo accuracy: 0.7143
```

The model performed better on the negative class than the positive class:

| Class | Meaning | Precision | Recall | F1-score | Support |
|---|---|---:|---:|---:|---:|
| 0 | non-diabetic | 0.76 | 0.82 | 0.79 | 100 |
| 1 | diabetic | 0.61 | 0.52 | 0.56 | 54 |

This means the model is better at detecting non-diabetic cases than diabetic cases. In a healthcare context, the lower recall for class `1` is important because false negatives can be risky.

## 12. Current SHAP Feature Ranking

The current demo produced this SHAP ranking:

| Rank | Feature | Mean Absolute SHAP |
|---:|---|---:|
| 1 | Glucose | 0.950060 |
| 2 | BMI | 0.528627 |
| 3 | Pregnancies | 0.337810 |
| 4 | DiabetesPedigreeFunction | 0.183460 |
| 5 | BloodPressure | 0.180990 |
| 6 | Age | 0.137601 |
| 7 | Insulin | 0.101974 |
| 8 | SkinThickness | 0.057666 |

`Glucose` is the most influential feature in the Logistic Regression model. This is medically reasonable because glucose level is directly related to diabetes diagnosis and diabetes risk.

## 13. Code Explanation: `shap_demo.py`

The file `shap_demo.py` is the runnable demo script. Its job is to reproduce the whole SHAP workflow.

Important code examples:

```python
data = pd.read_csv("dataset/diabetes.csv")
```

This loads the diabetes dataset.

```python
x = data.drop(target, axis=1)
y = data[target]
```

This separates input features from the prediction target.

```python
model = LogisticRegression(random_state=100, max_iter=1000)
model.fit(x_train_scaled, y_train)
```

This trains the Logistic Regression model.

```python
shap_result = apply_shap(
    model,
    x_train_scaled,
    x_test_scaled,
    feature_names=x.columns,
    output_dir="shap_outputs/logistic_regression_demo",
    plot_name="logistic_regression_demo",
)
```

This applies SHAP and exports the plots and CSV.

## 14. Code Explanation: `shap_explainability.py`

The file `shap_explainability.py` contains the reusable function `apply_shap(...)`.

Important code examples:

```python
estimator = _unwrap_estimator(model)
```

This allows the function to work with normal trained models and `GridSearchCV` models. If a `GridSearchCV` object has `best_estimator_`, the best estimator is used automatically.

```python
x_train_df = _as_dataframe(x_train, feature_names)
x_test_df = _as_dataframe(x_test, feature_names)
```

This converts NumPy arrays into pandas DataFrames so the SHAP plots can show real feature names.

```python
explainer = _build_explainer(shap, estimator, background)
shap_output = _calculate_shap_values(explainer, x_explain)
shap_values = _select_class_values(shap_output, class_index=class_index)
```

This builds the SHAP explainer, calculates SHAP values, and selects the class to explain. In this project, `class_index=1` explains the positive diabetes class.

```python
feature_importance = pd.DataFrame({
    "feature": x_explain.columns,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
})
```

This creates the feature-importance table. A larger `mean_abs_shap` value means the feature has a stronger average effect on model predictions.

## 15. Report Interpretation

The SHAP results show that `Glucose`, `BMI`, and `Pregnancies` were the most important features for this Logistic Regression diabetes prediction model. This is reasonable because glucose level and body mass index are commonly associated with diabetes risk.

The model achieved about 71% accuracy, but its recall for the positive diabetes class was lower than for the negative class. This suggests that the model may miss some diabetic cases. For a healthcare-related model, this is an important limitation and should be improved before real-world use.

Overall, SHAP improves the project because it adds interpretability. Instead of only saying the model predicted diabetes, we can explain which features contributed most strongly to the prediction.

## 16. How To Run This Notebook Or Demo Yourself

From the project root, activate the virtual environment:

```bash
source .venv/bin/activate
```

Then run the demo script:

```bash
python shap_demo.py
```

If Matplotlib gives a cache warning, run:

```bash
MPLCONFIGDIR=/private/tmp/mplconfig python shap_demo.py
```

The output files will be saved in:

```text
shap_outputs/logistic_regression_demo/
```

## 17. Final Report Paragraph

This project applies SHAP explainability to a diabetes prediction model. A Logistic Regression model was trained on the diabetes dataset, and SHAP was used to explain which clinical features influenced the model's predictions. The model achieved an accuracy of approximately 71.43% on the test set. SHAP analysis showed that `Glucose` was the most influential feature, followed by `BMI`, `Pregnancies`, and `DiabetesPedigreeFunction`. These features are medically meaningful for diabetes risk prediction. The SHAP helper function also exports a summary plot, bar plot, and feature-importance CSV file, making the model easier to interpret and report. However, the SHAP results explain the model's behavior and should not be interpreted as proof of medical causality.